# Uplift Modeling for Customer Churn — Full Pipeline

**Goal:** Identify which customers will respond positively to a retention offer  
(i.e., stop churning *because* of the offer), rather than simply customers most  
likely to churn regardless.

**Models:** S-Learner · T-Learner · X-Learner (all XGBoost-based)  
**Data:** IBM Telco Customer Churn (public dataset, ~7 k rows)  
**Treatment:** Synthetically simulated — see `src/preprocess.py` for assumptions

---
Run cells top-to-bottom. In Colab the first two sections handle environment setup.

## 1 — Setup

In [ ]:
# ── Colab: mount Drive and set repo path ───────────────────────────────────
import os, sys, pathlib

REPO_NAME  = "uplift-churn"            # folder name on your Drive root
DRIVE_ROOT = "/content/drive/MyDrive"

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_PATH = f"{DRIVE_ROOT}/{REPO_NAME}"
    if not os.path.isdir(REPO_PATH):
        raise FileNotFoundError(
            f"Folder not found: {REPO_PATH}\n"
            f"Upload the '{REPO_NAME}' folder to your Drive root and re-run."
        )
    sys.path.insert(0, REPO_PATH)
    os.chdir(REPO_PATH)

    DRIVE_OUT = pathlib.Path(REPO_PATH) / "outputs"
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    DRIVE_OUT = None

print('Working directory:', os.getcwd())
print('Python path entry:', sys.path[0])
if DRIVE_OUT:
    print('Outputs dir:', DRIVE_OUT)


In [ ]:
# ── Colab: install dependencies ─────────────────────────────────────────────
# Uses requirements-colab.txt (packages Colab doesn't pre-install).
# Full pinned versions are in requirements.txt for local dev.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess, sys
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError('pip install failed — see output above')
    print('Dependencies installed.')
else:
    print('Local run — make sure your venv is activated and requirements installed.')


In [ ]:
# Drive is already mounted and DRIVE_OUT is already set in the setup cell above.
# This cell is kept as a placeholder — nothing to do here.
print('DRIVE_OUT:', DRIVE_OUT)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from src import ingest, preprocess, models as m_module, evaluate

print('All imports OK.')

---
## 1b — Smoke Tests

Run the test suite before executing the pipeline to verify the environment is correct.

In [ ]:
# Runs pytest and streams output inline.
# Expected: all tests pass in ~60 s (S-Learner on 1 k-row synthetic fixture).
!pytest tests/ -v --tb=short --timeout=120

---
## 2 — Data Ingestion

In [ ]:
raw_df = ingest.load_raw()
raw_df.head(3)

In [ ]:
print(f"Shape: {raw_df.shape}")
print(f"\nChurn rate: {raw_df['Churn'].value_counts(normalize=True).to_dict()}")
raw_df.dtypes.value_counts()

In [ ]:
print('Missing values:')
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0])

---
## 3 — Preprocessing

Steps: clean dtypes → engineer features → label-encode categoricals → **add synthetic treatment column**.

> **Assumption**: The Telco dataset has no real randomised treatment. We simulate a binary
> `received_retention_offer` flag with a 40 % treatment probability, drawn independently
> of all features. This mimics an RCT pilot. See `src/preprocess.py` for the exact logic.

In [ ]:
processed_df, (X_train, X_test, y_train, y_test, w_train, w_test) = preprocess.run(raw_df)
processed_df.head(3)

In [ ]:
print('Treatment balance (train):')
print(w_train.value_counts(normalize=True).rename({0: 'Control', 1: 'Treated'}).to_string())
print(f'\nFeature matrix: {X_train.shape[1]} features')

---
## 4 — Modeling

We train three meta-learners using XGBoost base estimators:

| Model | Idea |
|-------|------|
| **S-Learner** | One model; treatment is a feature. Simple but biased when treatment effect is small. |
| **T-Learner** | Separate models for treated / control. High variance with small groups. |
| **X-Learner** | Cross-fits CATE estimates. Best for imbalanced T/C ratios. |

In [ ]:
fitted_models = m_module.train_all(X_train, y_train, w_train)

In [ ]:
cate_preds = m_module.predict_all(fitted_models, X_test)

for name, cate in cate_preds.items():
    print(f"{name:12s}  mean CATE={cate.mean():.4f}  std={cate.std():.4f}  "
          f"min={cate.min():.4f}  max={cate.max():.4f}")

---
## 5 — Evaluation

### 5a — Qini Curves and AUUC

The **Qini curve** plots the *incremental* number of churn cases prevented as we  
contact an increasing fraction of customers (ranked by predicted uplift).  
A good uplift model should curve well above the random diagonal.  
The **AUUC** (Area Under the Uplift Curve) condenses this into a single number.

In [ ]:
auuc_scores = evaluate.plot_qini_curves(cate_preds, y_test, w_test, drive_dir=DRIVE_OUT)
print('\nAUUC scores:')
for name, score in sorted(auuc_scores.items(), key=lambda x: -x[1]):
    print(f'  {name}: {score:.4f}')

### 5b — SHAP Feature Importance (Best Model)

SHAP values decompose each prediction into per-feature contributions, making  
the model interpretable. We explain the treatment-group learner of the best model.

In [ ]:
best_name = max(auuc_scores, key=auuc_scores.get)
best_model = next(m for m in fitted_models if m.name == best_name)
print(f'Explaining: {best_name}')

shap_vals = evaluate.explain_best_model(best_model, X_train, X_test, drive_dir=DRIVE_OUT)

### 5c — Fairness Disparity Report

We check whether predicted uplift differs systematically across `gender` and `SeniorCitizen` subgroups.  
A **disparity ratio** close to 1.0 indicates equitable targeting; values above ~1.2 warrant review.

In [ ]:
fairness_df = evaluate.fairness_report(X_test, cate_preds, drive_dir=DRIVE_OUT)
fairness_df

---
## 6 — Business ROI Analysis

Given:
- Contact cost = **\$5** per customer reached
- Revenue lost per churned customer = **\$200**

We compute the **Net Lift** if we target the top 10 %, 20 %, or 30 % of customers  
ranked by predicted uplift:

```
Net Lift = (avg_CATE × n_targeted × $200) − (n_targeted × $5)
```

Break-even CATE threshold: $5 / $200 = **0.025** — only target customers above this.

In [ ]:
CONTACT_COST = 5.0
CHURN_REVENUE = 200.0

roi_df = evaluate.roi_table(
    cate_preds, y_test, w_test,
    contact_cost=CONTACT_COST,
    churn_revenue=CHURN_REVENUE,
    percentiles=(0.10, 0.20, 0.30),
    drive_dir=DRIVE_OUT,
)

thresh = m_module.optimal_threshold(CONTACT_COST, CHURN_REVENUE)
print(f'\nBreak-even CATE threshold: {thresh:.4f}')
for name, cate in cate_preds.items():
    frac = m_module.optimal_targeting_fraction(cate, CONTACT_COST, CHURN_REVENUE)
    print(f'  {name}: target {frac:.1%} of customers')

roi_df

---
## 7 — MLflow Experiment Tracking

One MLflow run is logged per model, capturing hyperparameters, AUUC, and the optimal
targeting threshold. The `mlruns/` folder is synced to Google Drive for persistence.

In [ ]:
import os, pathlib

MLRUNS_DIR = str(pathlib.Path(os.getcwd()) / 'mlruns')

run_ids = m_module.log_all_experiments(
    fitted_models,
    auuc_scores,
    cate_preds,
    contact_cost=CONTACT_COST,
    churn_revenue=CHURN_REVENUE,
    tracking_uri=MLRUNS_DIR,
)
print('\nMLflow run IDs:', run_ids)
print('Tracking URI:', MLRUNS_DIR)


In [ ]:
# mlruns/ is written directly into the repo folder (Drive or local),
# so no copy step is needed.
import pathlib, os
mlruns_path = pathlib.Path(os.getcwd()) / 'mlruns'
if mlruns_path.exists():
    print(f'MLflow runs stored at: {mlruns_path}')
    print(f'View locally with: mlflow ui --backend-store-uri "{mlruns_path}"')
else:
    print('mlruns/ not found — check that the mlflow-log cell ran without errors.')


---
## Summary

| Artifact | Location |
|----------|----------|
| Qini curves | `outputs/figures/qini_curves.png` |
| SHAP bar | `outputs/figures/shap_summary.png` |
| SHAP beeswarm | `outputs/figures/shap_beeswarm.png` |
| ROI chart | `outputs/figures/roi_net_lift.png` |
| ROI table (CSV) | `outputs/roi_table.csv` |
| Fairness report | `outputs/fairness_report.csv` |
| MLflow runs | `mlruns/` (and Drive when in Colab) |

Open `notebooks/colab_dashboard.ipynb` for the interactive version.